In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

# 1. Khởi tạo Spark Session (Không dùng HiveSupport)
spark = SparkSession.builder \
    .appName("Task 1 - Delivery Ingest Bronze") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc file CSV từ MinIO
path = "s3a://lakehouse/delivery_yt.csv"
df = spark.read.csv(path, header=True, inferSchema=True)

# 3. Thống kê
print(f"Tổng số bản ghi (rows): {df.count()}")
print("=== THỐNG KÊ GIÁ TRỊ NULL THỜI GIAN/KHU VỰC ===")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 4. Ghi trực tiếp ra S3 path (Bỏ qua saveAsTable)
print("Đang tiến hành ghi dữ liệu vào Delta Lake...")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://lakehouse/warehouse/delivery_bronze")

print("Đã lưu bảng 'delivery_bronze' thành công!")

Tổng số bản ghi (rows): 206431
=== THỐNG KÊ GIÁ TRỊ NULL THỜI GIAN/KHU VỰC ===
+--------+---------+----+----------+---+---+------+--------+-----------+---------------+--------------+--------------+-------------+-----------------+----------------+----------------+---+
|order_id|region_id|city|courier_id|lng|lat|aoi_id|aoi_type|accept_time|accept_gps_time|accept_gps_lng|accept_gps_lat|delivery_time|delivery_gps_time|delivery_gps_lng|delivery_gps_lat| ds|
+--------+---------+----+----------+---+---+------+--------+-----------+---------------+--------------+--------------+-------------+-----------------+----------------+----------------+---+
|       0|        0|   0|         0|  0|  0|     0|       0|          0|              0|             0|             0|            0|                0|               0|               0|  0|
+--------+---------+----+----------+---+---+------+--------+-----------+---------------+--------------+--------------+-------------+-----------------+---------------